# Query-Representation Ablation

**Question.** Do the MNRL retriever's gains persist when the query contains only what will actually be available at generation time — i.e. **no distractor**, because the distractor is the thing Stage 2 is trying to produce?

**Design — one independent variable: the query text.** The model checkpoint, corpus, split, retrieval algorithm, metrics, and evaluation code are all held fixed. **No retraining.** The corpus is encoded **once** and reused byte-identically across all variants, so any difference is attributable to the query representation alone.

| Variant | Query fields |
|---|---|
| **C** (reference) | Subject + Construct + Question + Correct Answer + Distractor |
| **B** | Subject + Construct + Question + Correct Answer |
| **A** (deployment-faithful) | Question + Correct Answer |

The corpus always uses the full template. That is deliberate and deployment-faithful: historical corpus items *do* have known distractors; an incoming question does not. The deployed setting is genuinely asymmetric.

**Only edit `REPO_URL`, set Runtime → GPU, then Runtime → Run all.** Expect ~20–25 min, almost all of it reproducing the MNRL checkpoint.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this

# If you already have a trained MNRL model saved (e.g. on Drive), set its path
# here to skip the ~15 min re-train. Leave as None to train seed 42 fresh.
EXISTING_MODEL = None   # e.g. "/content/drive/MyDrive/finetuned_mnrl"
SEED = 42

In [ ]:
# --- Verify GPU ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repo and install pinned deps ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
assert os.path.exists("datasets/train.csv"), "datasets/train.csv missing from repo"
print("Dataset present.")

In [ ]:
# --- Preprocessing: splits and triplets (identical to all prior experiments) ---
import os
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    !python scripts/03_create_triplets.py
print("Preprocessing ready.")

In [ ]:
# --- Obtain the MNRL checkpoint under test ---
# The multi-seed study's models were never downloaded, so unless you supply an
# existing path we reproduce the seed-42 MNRL model with the same code and seed.
# This is a REPRODUCTION, not a new training condition. All three query variants
# are then evaluated against this ONE checkpoint, which is what makes the
# comparison clean — Variant C is recomputed here rather than copied from the
# multi-seed table, so every variant shares an identical model.
import os
MODEL_DIR = EXISTING_MODEL or "outputs/models/finetuned_mnrl"
if EXISTING_MODEL is None and not os.path.exists(f"{MODEL_DIR}/config.json"):
    !python scripts/train_gpu.py --objective mnrl --seed {SEED} \
      --save-steps 50 --eval-steps 50 --keep-all-checkpoints \
      --output {MODEL_DIR}
else:
    print(f"Using existing model at {MODEL_DIR}")
print("Model ready:", MODEL_DIR)

In [ ]:
# --- Run the ablation ---
# --baseline  : also score the pretrained encoder under each query variant, so
#               "does fine-tuning still help?" can be answered in the deployment
#               setting rather than only under the full query.
# --symmetric : supplementary diagnostic that also reduces the CORPUS template,
#               separating information loss from train/inference format mismatch.
!python scripts/07_query_ablation.py --model {MODEL_DIR} --baseline --symmetric

In [ ]:
# --- Results table ---
import pandas as pd
df = pd.read_csv("outputs/results/query_ablation/query_ablation_results.csv")
cols = ["variant", "model", "query_representation",
        "MRR@10", "HitRate@10", "nDCG@10", "Recall@10"]
display(df[cols].round(4))

In [ ]:
# --- Headline comparison: retention relative to the reference variant ---
m = df[(df.model == "MNRL") & (~df.variant.str.contains("-sym"))].set_index("variant")
C, B, A = m.loc["C"], m.loc["B"], m.loc["A"]
print(f"{'':38s} {'MRR@10':>9s} {'Δ vs C':>9s} {'% of C':>8s}")
for lab, row in [("C  full (reference)", C),
                 ("B  no distractor", B),
                 ("A  question + answer (deployment)", A)]:
    d = row['MRR@10'] - C['MRR@10']
    print(f"{lab:38s} {row['MRR@10']:9.4f} {d:+9.4f} {row['MRR@10']/C['MRR@10']*100:7.1f}%")
print(f"\nDistractor contribution (C - B): {C['MRR@10']-B['MRR@10']:+.4f} MRR@10")
print(f"Metadata contribution   (B - A): {B['MRR@10']-A['MRR@10']:+.4f} MRR@10")

In [ ]:
# --- Paired per-query significance vs. the reference variant ---
sig = pd.read_csv("outputs/results/query_ablation/query_ablation_significance.csv")
display(sig)

In [ ]:
# --- Figure ---
from IPython.display import Image, display as disp
disp(Image("outputs/figures/query_ablation/query_ablation.png"))

In [ ]:
# --- Auto-generated report ---
print(open("outputs/results/query_ablation/query_ablation_report.md").read())

## How to read the result

Compare **Variant A** (deployment-faithful) against **Variant C** (current reference), using MRR@10 retention as the headline. All three share one checkpoint and one corpus, so the difference is the query representation and nothing else.

**A retains ≥ 90% of C — the current retriever is validated.** The distractor was not doing the heavy lifting; the retriever keys on the question and correct answer. Carry it into Stage 2 unchanged. Report the question-only numbers as the operative figures, since those are what generation will actually see.

**A retains 70–90% — partial dependence.** The retriever still works but a real share of its performance came from the distractor. Usable for Stage 2, but retraining on question-only inputs is advisable and would likely recover part of the gap. Report both numbers in the thesis.

**A retains < 70% — the distractor was carrying most of the signal.** The Stage 1 headline numbers substantially overstate what generation will get. Retraining with question-only inputs is required before Stage 2 integration, and the Stage 1 results must be reported with this caveat attached prominently.

**Also check, whichever bucket you land in:**

- **Does fine-tuning still beat the pretrained baseline under Variant A?** This is the question that actually matters for Stage 2. If MNRL under A no longer beats the baseline under A, the fine-tuning provides no deployable benefit regardless of what the full-query numbers showed — that would be the single most important finding of this ablation.
- **A vs. B** isolates the distractor's contribution; **B vs. A** isolates the metadata's. If metadata contributes strongly, note that Subject/Construct may also be unavailable for a genuinely new question, so Variant A remains the honest deployment estimate.
- **A vs. A-sym** (supplementary) separates the two causes of any drop. A large gap implicates train/inference format mismatch, which retraining can fix. A small gap implicates genuine information loss, which retraining cannot fix — that would call for rethinking the task framing rather than the training recipe.

**Caveat this ablation cannot resolve:** the model was trained on full-QDP text and has never seen question-only input. A drop therefore conflates information loss with distribution shift. That is precisely why retraining is the *conditional* follow-up rather than something done pre-emptively here.

In [ ]:
# --- Package and download ---
!zip -qr query_ablation_results.zip \
  outputs/results/query_ablation \
  outputs/figures/query_ablation
from google.colab import files
files.download("query_ablation_results.zip")